<a href="https://colab.research.google.com/github/Aa-sheesh/Aa-sheesh/blob/master/basic_lung_cancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

# Download the dataset
path = kagglehub.dataset_download("mohamedhanyyy/chest-ctscan-images")
print("Path to dataset files:", path)

import random, shutil, glob, warnings
warnings.simplefilter('ignore')
from matplotlib import pyplot
from matplotlib.image import imread

import keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, InputLayer, BatchNormalization
from keras import optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import InceptionV3, ResNet50
from sklearn import metrics

# Define directories
train_data = os.path.join(path, 'Data/train')
test_data = os.path.join(path, 'Data/test')
validation_data = os.path.join(path, 'Data/valid')

# Data generators for training, validation, and testing
train_datagen = ImageDataGenerator(
    dtype='float32',
    rotation_range=10,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=False,
)
test_datagen = ImageDataGenerator(
    dtype='float32',
    rotation_range=10,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=False,
)
validation_datagen = ImageDataGenerator(
    dtype='float32',
    rotation_range=10,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=False,
)

train_generator = train_datagen.flow_from_directory(
    train_data,
    target_size=(224, 224),
    batch_size=64,
    class_mode='categorical',
)

test_generator = test_datagen.flow_from_directory(
    test_data,
    target_size=(224, 224),
    batch_size=200,
    class_mode='categorical'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_data,
    target_size=(224, 224),
    batch_size=64,
    class_mode='categorical',
)

# Optionally display sample images from each class
train_normal = glob.glob(os.path.join(train_data, 'Normal/*'))
train_adenocarcinoma = glob.glob(os.path.join(train_data, 'Adenocarcinoma/*'))
train_squamous = glob.glob(os.path.join(train_data, 'Squamous/*'))
train_large_cell_carcinoma = glob.glob(os.path.join(train_data, 'Large cell carcinoma/*'))

fig1, ax1 = plt.subplots(1, 5, figsize=(15, 4))
fig1.suptitle("Normal", fontsize=18)
for i in range(min(5, len(train_normal))):
    ax1[i].imshow(imread(train_normal[i]))

fig1, ax1 = plt.subplots(1, 5, figsize=(15, 4))
fig1.suptitle("Adenocarcinoma", fontsize=18)
for i in range(min(5, len(train_adenocarcinoma))):
    ax1[i].imshow(imread(train_adenocarcinoma[i]))

fig1, ax1 = plt.subplots(1, 5, figsize=(15, 4))
fig1.suptitle("Squamous", fontsize=18)
for i in range(min(5, len(train_squamous))):
    ax1[i].imshow(imread(train_squamous[i]))

fig1, ax1 = plt.subplots(1, 5, figsize=(15, 4))
fig1.suptitle("Large Cell Carcinoma", fontsize=18)
for i in range(min(5, len(train_large_cell_carcinoma))):
    ax1[i].imshow(imread(train_large_cell_carcinoma[i]))

plt.show()

# ---------------------------
# Model Construction & Training
# ---------------------------
image_shape = (224, 224, 3)
res_model = ResNet50(include_top=False, pooling='avg', weights='imagenet', input_shape=image_shape)

# Freeze all layers except those in the conv5 block
for layer in res_model.layers:
    if 'conv5' not in layer.name:
        layer.trainable = False

resnet_model = Sequential()
resnet_model.add(res_model)
resnet_model.add(Dropout(0.3))
resnet_model.add(Flatten())
resnet_model.add(BatchNormalization())
resnet_model.add(Dropout(0.3))
resnet_model.add(Dense(256, activation='relu'))
resnet_model.add(BatchNormalization())
resnet_model.add(Dropout(0.3))
resnet_model.add(Dense(4, activation='softmax'))

optimizer = optimizers.SGD(learning_rate=0.00003)
resnet_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['acc'])
resnet_model.summary()

history_res = resnet_model.fit(
    train_generator,
    steps_per_epoch=8,
    epochs=70,
    verbose=1,
    validation_data=validation_generator
)

# ---------------------------
# Save the trained model for deployment
# ---------------------------
resnet_model.save('lung_cancer_model.h5')
print("Model saved as lung_cancer_model.h5")
